# Phase 4-5 学习指南：风险分析与市场概览

**核心理念：** 风险管理比预测更重要。在学会预测之前，先学会评估风险。

---
# Phase 4 — Risk Analysis

风险分析是投资中最被低估的技能。很多策略在赚钱一段时间后最终爆仓，就是因为忽视风险。

## 1. 收益率计算

### 日收益率（Daily Returns）

```
日收益率 = (今日价格 - 昨日价格) / 昨日价格
         = 今日价格 / 昨日价格 - 1
```

在 pandas 中通常用 `pct_change()` 计算。

**注意：** 用 Adj Close 计算，不是 Close

In [ ]:
import yfinance as yf
import pandas as pd

# 下载 SPY 数据
spy = yf.download("SPY", period="1y")

# 计算日收益率
spy['Daily_Return'] = spy['Adj Close'].pct_change()

print("最近5天的日收益率：")
print(spy['Daily_Return'].tail())

# 日收益率统计
print(f"\n平均日收益: {spy['Daily_Return'].mean()*100:.3f}%")
print(f"日收益标准差: {spy['Daily_Return'].std()*100:.3f}%")
print(f"最大单日涨幅: {spy['Daily_Return'].max()*100:.2f}%")
print(f"最大单日跌幅: {spy['Daily_Return'].min()*100:.2f}%")

### 年化收益率（Annualized Return）

日收益率太小不容易理解，通常换算成年化：

```
年化收益率 = (1 + 总收益率)^(252/交易日数) - 1
```

为什么是 252？一年大约 252 个交易日。

In [ ]:
# 计算年化收益率
total_days = len(spy)
total_return = (spy['Adj Close'].iloc[-1] / spy['Adj Close'].iloc[0]) - 1
annualized_return = (1 + total_return) ** (252 / total_days) - 1

print(f"持有天数: {total_days}")
print(f"总收益率: {total_return*100:.2f}%")
print(f"年化收益率: {annualized_return*100:.2f}%")

## 2. 波动率（Volatility）

波动率 = 收益率的标准差，衡量价格波动的剧烈程度。

```
日波动率 = 日收益率的标准差
年化波动率 = 日波动率 × √252
```

**为什么乘 √252？**
- 方差的累加性：方差可以相加
- 252 天的方差 = 日方差 × 252
- 标准差 = √方差 = 日标准差 × √252

In [ ]:
import numpy as np

# 计算波动率
daily_vol = spy['Daily_Return'].std()
annualized_vol = daily_vol * np.sqrt(252)

print(f"日波动率: {daily_vol*100:.3f}%")
print(f"年化波动率: {annualized_vol*100:.2f}%")

# 对比不同资产
tickers = ["SPY", "QQQ", "IWM", "GLD", "TLT"]
print("\n不同资产的年化波动率对比：")
for ticker in tickers:
    df = yf.download(ticker, period="1y", progress=False)
    vol = df['Adj Close'].pct_change().std() * np.sqrt(252)
    print(f"{ticker}: {vol*100:.2f}%")

## 3. 最大回撤（Max Drawdown）

回撤 = 从最高点到最低点的跌幅
最大回撤 = 历史上最大的回撤幅度

**为什么重要？**
- 告诉你最坏的情况下会亏多少
- 检验你能不能承受这个亏损
- 很多策略的致命伤就是一次大回撤

**计算步骤：**
1. 计算累计最高点（rolling max）
2. 计算回撤 = (当前价格 - 累计最高点) / 累计最高点
3. 最大回撤 = 最小的回撤值（负数，取绝对值）

In [ ]:
# 计算最大回撤
spy['Cummax'] = spy['Adj Close'].cummax()  # 累计最高点
spy['Drawdown'] = (spy['Adj Close'] - spy['Cummax']) / spy['Cummax']
max_drawdown = spy['Drawdown'].min()

print(f"最大回撤: {max_drawdown*100:.2f}%")

# 找到最大回撤发生的时间
max_dd_date = spy['Drawdown'].idxmin()
print(f"最大回撤发生日期: {max_dd_date.strftime('%Y-%m-%d')}")

# 可视化回撤
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(spy.index, spy['Adj Close'])
axes[0].set_title('SPY Price')
axes[0].set_ylabel('Price ($)')

axes[1].fill_between(spy.index, spy['Drawdown'], 0, alpha=0.3, color='red')
axes[1].set_title('Drawdown')
axes[1].set_ylabel('Drawdown (%)')
axes[1].set_xlabel('Date')

plt.tight_layout()
plt.show()

## 4. 夏普比率（Sharpe Ratio）

最常用的风险调整收益指标。

```
夏普比率 = (年化收益率 - 无风险利率) / 年化波动率
```

**解读：**
| 夏普比率 | 评价 |
|----------|------|
| < 0 | 亏损 |
| 0-1 | 收益不足以补偿风险 |
| 1-2 | 好 |
| 2-3 | 很好 |
| > 3 | 优秀（或数据有问题）|

**注意：** 夏普比率假设收益正态分布，实际市场有肥尾效应。

In [ ]:
# 计算夏普比率
risk_free_rate = 0.05  # 假设无风险利率 5%

sharpe = (annualized_return - risk_free_rate) / annualized_vol

print(f"年化收益: {annualized_return*100:.2f}%")
print(f"年化波动: {annualized_vol*100:.2f}%")
print(f"夏普比率: {sharpe:.2f}")

## 5. Beta

Beta 衡量个股相对于基准（通常是 SPY）的波动敏感度。

```
Beta = Cov(股票收益, 市场收益) / Var(市场收益)
```

**解读：**
| Beta | 含义 |
|------|------|
| Beta = 1 | 和市场同步 |
| Beta > 1 | 比市场更波动（放大收益和损失）|
| Beta < 1 | 比市场更稳定 |
| Beta < 0 | 与市场反向（如做空策略）|

In [ ]:
# 计算 Beta
def calculate_beta(ticker, market="SPY", period="1y"):
    """计算股票相对于市场的 Beta"""
    # 下载数据
    stock = yf.download(ticker, period=period, progress=False)
    mkt = yf.download(market, period=period, progress=False)
    
    # 对齐日期
    stock = stock['Adj Close'].pct_change().dropna()
    mkt = mkt['Adj Close'].pct_change().dropna()
    aligned = pd.concat([stock, mkt], axis=1, join='inner').dropna()
    
    # 计算 Beta
    cov = aligned.cov().iloc[0, 1]
    var = aligned.iloc[:, 1].var()
    beta = cov / var
    
    return beta

# 对比不同股票的 Beta
stocks = ["AAPL", "MSFT", "TSLA", "JNJ", "XOM"]
print("各股票相对于 SPY 的 Beta：")
for stock in stocks:
    beta = calculate_beta(stock)
    print(f"{stock}: {beta:.2f}")

## 6. 相关性矩阵（Correlation Matrix）

衡量不同资产之间的联动程度。

- 相关系数 = 1：完全同步
- 相关系数 = 0：无关联
- 相关系数 = -1：完全反向

**用途：** 分散投资 — 选择相关性低的资产组合

In [ ]:
# 计算相关性矩阵
import seaborn as sns

tickers = ["SPY", "QQQ", "IWM", "GLD", "TLT", "VIX"]

# 下载所有数据
data = yf.download(tickers, period="1y", progress=False)['Adj Close']
returns = data.pct_change().dropna()

# 相关性矩阵
corr_matrix = returns.corr()

print("收益率相关性矩阵：")
print(corr_matrix.round(2))

# 热力图
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='RdYlGn', center=0, fmt='.2f')
plt.title('Asset Correlation Matrix')
plt.show()

print("\n解读：")
print("- SPY/QQQ/IWM 高度相关（美国股市）")
print("- GLD 与股票相关性低（黄金作为分散）")
print("- TLT 与股票负相关（债券 vs 股票）")
print("- VIX 与股票负相关（恐慌指数）")

## 7. 风险指标汇总函数

把所有指标整合成一个函数，方便使用。

In [ ]:
def calculate_risk_metrics(ticker, period="2y", risk_free_rate=0.05):
    """
    计算股票的所有风险指标
    
    Returns:
        dict: 包含各项风险指标
    """
    df = yf.download(ticker, period=period, progress=False)
    
    # 收益率
    df['Return'] = df['Adj Close'].pct_change()
    
    # 总收益率和年化收益率
    total_return = df['Adj Close'].iloc[-1] / df['Adj Close'].iloc[0] - 1
    days = len(df)
    ann_return = (1 + total_return) ** (252 / days) - 1
    
    # 波动率
    daily_vol = df['Return'].std()
    ann_vol = daily_vol * np.sqrt(252)
    
    # 最大回撤
    cummax = df['Adj Close'].cummax()
    drawdown = (df['Adj Close'] - cummax) / cummax
    max_dd = drawdown.min()
    
    # 夏普比率
    sharpe = (ann_return - risk_free_rate) / ann_vol
    
    # 下行波动率（只看负收益）
    downside_returns = df['Return'][df['Return'] < 0]
    downside_vol = downside_returns.std() * np.sqrt(252)
    
    # Sortino 比率（用下行波动率替代总波动率）
    sortino = (ann_return - risk_free_rate) / downside_vol
    
    return {
        'Ticker': ticker,
        'Total_Return': f"{total_return*100:.2f}%",
        'Annualized_Return': f"{ann_return*100:.2f}%",
        'Annualized_Volatility': f"{ann_vol*100:.2f}%",
        'Max_Drawdown': f"{max_dd*100:.2f}%",
        'Sharpe_Ratio': f"{sharpe:.2f}",
        'Sortino_Ratio': f"{sortino:.2f}"
    }

# 对比多个资产
tickers = ["SPY", "QQQ", "IWM", "GLD", "TLT"]
results = []

for ticker in tickers:
    results.append(calculate_risk_metrics(ticker))

pd.DataFrame(results)

## 自测问题（Phase 4）

1. 如果两个投资组合收益相同，一个波动率 10%，一个 20%，你选哪个？为什么？

2. 最大回撤 -30% 意味着什么？如果你投资 $10000，最坏情况下变成多少？

3. 夏普比率 0.8 和 1.5 的区别是什么？

4. 为什么 Sortino 比率比夏普比率更合理？

5. Beta = 1.5 的股票在市场涨 10% 时预期涨多少？市场跌 10% 时呢？

6. 你要构建一个分散的投资组合，应该选择高相关性还是低相关性的资产？

---
# Phase 5 — Market Overview Dashboard

建立对整体市场的感知能力。

## 1. 市场概览的核心指标

### 主要指数 ETF

| Ticker | 代表 | 市场含义 |
|--------|------|----------|
| SPY | 标普500 | 大盘股整体 |
| QQQ | 纳斯达克100 | 科技股 |
| DIA | 道琼斯 | 传统蓝筹 |
| IWM | 罗素2000 | 小盘股 |
| VIX | 波动率指数 | 市场恐慌程度 |

### 板块 ETF（Sector ETFs）

| Sector | ETF |
|--------|-----|
| 科技 | XLK |
| 金融 | XLF |
| 医疗 | XLV |
| 能源 | XLE |
| 消费 | XLY |
| 工业 | XLI |
| 材料 | XLB |
| 公用事业 | XLU |
| 房地产 | XLRE |
| 必需消费 | XLP |

In [ ]:
# 下载主要指数数据
major_etfs = ["SPY", "QQQ", "DIA", "IWM"]
data = yf.download(major_etfs, period="1y", progress=False)['Adj Close']

# 归一化对比
normalized = data / data.iloc[0] * 100

normalized.plot(figsize=(12, 6))
plt.title('Major Index ETFs Comparison (Normalized)')
plt.ylabel('Normalized Price (Start = 100)')
plt.legend(major_etfs)
plt.grid(True, alpha=0.3)
plt.show()

# 计算各指数表现
returns = (data.iloc[-1] / data.iloc[0] - 1) * 100
print("过去一年表现：")
for etf in major_etfs:
    print(f"{etf}: {returns[etf]:.2f}%")

## 2. 板块轮动（Sector Rotation）

不同经济周期，表现最好的板块不同：

| 经济周期 | 强势板块 |
|----------|----------|
| 复苏期 | 金融、工业、可选消费 |
| 扩张期 | 科技、通信 |
| 滞胀期 | 能源、材料 |
| 衰退期 | 必需消费、医疗、公用事业 |

In [ ]:
# 板块表现对比
sector_etfs = {
    'XLK': 'Technology',
    'XLF': 'Financials', 
    'XLV': 'Healthcare',
    'XLE': 'Energy',
    'XLY': 'Consumer Disc.',
    'XLP': 'Consumer Staples',
    'XLI': 'Industrials',
    'XLB': 'Materials',
    'XLU': 'Utilities',
    'XLRE': 'Real Estate'
}

# 下载数据
sector_data = yf.download(list(sector_etfs.keys()), period="3m", progress=False)['Adj Close']

# 计算近期表现
sector_returns = (sector_data.iloc[-1] / sector_data.iloc[0] - 1) * 100

# 排序
sector_returns_sorted = sector_returns.sort_values(ascending=True)

# 可视化
plt.figure(figsize=(10, 6))
colors = ['green' if x > 0 else 'red' for x in sector_returns_sorted]
plt.barh([sector_etfs[t] for t in sector_returns_sorted.index], 
         sector_returns_sorted.values, color=colors)
plt.axvline(x=0, color='black', linewidth=0.5)
plt.xlabel('Return (%)')
plt.title('Sector Performance (Last 3 Months)')
plt.grid(True, axis='x', alpha=0.3)
plt.show()

## 3. VIX 与市场情绪

VIX = 标普500期权的隐含波动率

**被称为「恐慌指数」**

| VIX 水平 | 市场状态 |
|----------|----------|
| < 15 | 非常平静，可能过度乐观 |
| 15-20 | 正常 |
| 20-30 | 开始紧张 |
| 30-40 | 恐慌 |
| > 40 | 极度恐慌（通常伴随大跌）|

In [ ]:
# VIX 与 SPY 的关系
vix = yf.download("^VIX", period="1y", progress=False)['Adj Close']
spy = yf.download("SPY", period="1y", progress=False)['Adj Close']

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(spy.index, spy, label='SPY')
axes[0].set_title('SPY Price')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(vix.index, vix, color='orange', label='VIX')
axes[1].axhline(y=20, color='red', linestyle='--', alpha=0.5, label='Panic Level (20)')
axes[1].set_title('VIX (Fear Index)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"当前 VIX: {vix.iloc[-1]:.2f}")
print(f"过去一年最高 VIX: {vix.max():.2f}")
print(f"过去一年最低 VIX: {vix.min():.2f}")

## 4. 市场宽度指标（Market Breadth）

市场涨跌不是所有股票同步的。市场宽度告诉我们：
- 多少股票在涨，多少在跌
- 上涨是普遍的还是少数股票带动的

**常用指标：**
- 涨跌比（Advancing/Declining Issues）
- 涨跌成交量比
- 新高新低比

**解读：**
- 指数涨 + 市场宽度好 = 健康上涨
- 指数涨 + 市场宽度差 = 少数大盘股拉升，可能不可持续

In [ ]:
# 模拟市场宽度计算
# 实际数据需要专业数据源（如 NYSE Adv/Dec）
# 这里用 S&P 500 成分股模拟

import random
random.seed(42)

# 模拟：生成随机涨跌数据
days = 20
advancing = [random.randint(200, 350) for _ in range(days)]
declining = [500 - a for a in advancing]

# 涨跌比
adv_dec_ratio = [a/d if d > 0 else a for a, d in zip(advancing, declining)]

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

x = range(days)
axes[0].bar(x, advancing, label='Advancing', color='green', alpha=0.7)
axes[0].bar(x, [-d for d in declining], bottom=[0]*days, label='Declining', color='red', alpha=0.7)
axes[0].axhline(y=0, color='black', linewidth=0.5)
axes[0].set_ylabel('Number of Stocks')
axes[0].set_title('Market Breadth: Advancing vs Declining Stocks')
axes[0].legend()

axes[1].plot(x, adv_dec_ratio, marker='o')
axes[1].axhline(y=1, color='red', linestyle='--', label='Neutral (1.0)')
axes[1].set_ylabel('Advance/Decline Ratio')
axes[1].set_xlabel('Days')
axes[1].set_title('A/D Ratio')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. 市场状态系统（Risk-On / Risk-Off）

简化版市场状态判断：

| 状态 | 特征 | 投资者行为 |
|------|------|-----------|
| **Risk-On** | VIX低、股市涨、资金流入股票 | 买入股票、高Beta资产 |
| **Neutral** | 信号混合 | 保持中性仓位 |
| **Risk-Off** | VIX高、股市跌、资金流入债券/黄金 | 降低仓位、避险 |

**简单规则示例：**
```
if VIX < 15 and SPY > MA50:
    status = "Risk-On"
elif VIX > 25 or SPY < MA200:
    status = "Risk-Off"
else:
    status = "Neutral"
```

In [ ]:
def get_market_regime():
    """简单的市场状态判断"""
    # 获取数据
    spy = yf.download("SPY", period="1y", progress=False)
    vix = yf.download("^VIX", period="1m", progress=False)
    
    # 计算均线
    spy['MA50'] = spy['Adj Close'].rolling(50).mean()
    spy['MA200'] = spy['Adj Close'].rolling(200).mean()
    
    # 当前值
    current_vix = vix['Adj Close'].iloc[-1]
    current_spy = spy['Adj Close'].iloc[-1]
    ma50 = spy['MA50'].iloc[-1]
    ma200 = spy['MA200'].iloc[-1]
    
    # 判断逻辑
    if current_vix < 15 and current_spy > ma50:
        regime = "RISK-ON 🟢"
        desc = "市场乐观，风险偏好高"
    elif current_vix > 25 or current_spy < ma200:
        regime = "RISK-OFF 🔴"
        desc = "市场恐慌，规避风险"
    else:
        regime = "NEUTRAL 🟡"
        desc = "市场中性，保持观望"
    
    return {
        'regime': regime,
        'description': desc,
        'VIX': f"{current_vix:.1f}",
        'SPY': f"{current_spy:.2f}",
        'SPY_vs_MA50': 'Above' if current_spy > ma50 else 'Below',
        'SPY_vs_MA200': 'Above' if current_spy > ma200 else 'Below'
    }

# 运行
regime = get_market_regime()
print("\n=== 市场状态 ===")
for key, value in regime.items():
    print(f"{key}: {value}")

## 自测问题（Phase 5）

1. QQQ 涨 5%，SPY 涨 2%，IWM 跌 1%，说明了什么市场特征？

2. 为什么 VIX 被称为「恐慌指数」？VIX = 35 意味着什么？

3. 板块轮动的逻辑是什么？为什么衰退期必需消费板块表现好？

4. 「指数涨但市场宽度差」意味着什么？

5. 你会如何利用 Risk-On / Risk-Off 状态调整你的投资策略？

---
## 学习检查清单

### Phase 4
- [ ] 理解日收益率和年化收益率的计算
- [ ] 理解波动率及其年化方法
- [ ] 会计算和解读最大回撤
- [ ] 理解夏普比率和 Sortino 比率
- [ ] 理解 Beta 的含义和计算
- [ ] 会使用相关性矩阵

### Phase 5
- [ ] 熟悉主要指数 ETF
- [ ] 理解板块轮动概念
- [ ] 理解 VIX 与市场情绪
- [ ] 理解市场宽度指标
- [ ] 能判断简单的 Risk-On / Risk-Off 状态

---
**完成后，切换到 Codex 执行 Phase 4-5 的代码实现。**